In [1]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import numpy as np

type_colors = ["#3498db", "#9b59b6", "#e74c3c", "#e67e22"]
all_types = impact_df["Disaster Type"].unique().tolist()
color_map = {t: type_colors[i % len(type_colors)] for i, t in enumerate(all_types)}

def plot_impact_dashboard(year_range):
    filtered = impact_df[
        (impact_df["Start Year"] >= year_range[0]) &
        (impact_df["Start Year"] <= year_range[1])
    ].copy()

    if filtered.empty:
        print("No data available for selected year range.")
        return

    filtered["Decade"] = (filtered["Start Year"] // 10 * 10).astype(str) + "s"

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(
        f"Impact Estimation Model\nEM-DAT Disaster Database ({year_range[0]}–{year_range[1]})",
        fontsize=14, fontweight="bold"
    )

    type_counts = filtered["Disaster Type"].value_counts()
    bar_colors_1 = [color_map[t] for t in type_counts.index]
    bars = axes[0, 0].bar(type_counts.index, type_counts.values,
                          color=bar_colors_1, edgecolor="white")
    axes[0, 0].set_title("Number of Disaster Events by Type")
    axes[0, 0].set_xlabel("Disaster Type")
    axes[0, 0].set_ylabel("Count")
    for bar, v in zip(bars, type_counts.values):
        axes[0, 0].text(bar.get_x() + bar.get_width()/2, v + max(type_counts.values) * 0.01,
                        str(v), ha="center", fontsize=10, fontweight="bold")

    avg_pop = (filtered.groupby("Disaster Type")["Total Affected"]
               .mean().sort_values(ascending=False))
    bar_colors_2 = [color_map[t] for t in avg_pop.index]
    bars2 = axes[0, 1].bar(avg_pop.index, avg_pop.values / 1e6,
                            color=bar_colors_2, edgecolor="white")
    axes[0, 1].set_title("Average Population Affected per Event")
    axes[0, 1].set_xlabel("Disaster Type")
    axes[0, 1].set_ylabel("Millions of People")
    for bar, v in zip(bars2, avg_pop.values):
        axes[0, 1].text(bar.get_x() + bar.get_width()/2, v / 1e6 + max(avg_pop.values) / 1e6 * 0.01,
                        f"{v/1e6:.1f}M", ha="center", fontsize=10, fontweight="bold")

    axes[1, 0].scatter(yp_test, pop_pred_log, alpha=0.25, s=14, color="#3498db")
    mn = min(yp_test.min(), pop_pred_log.min())
    mx = max(yp_test.max(), pop_pred_log.max())
    axes[1, 0].plot([mn, mx], [mn, mx], color="green", label="Perfect prediction")
    axes[1, 0].set_title(f"Actual vs Predicted — Population Affected\n(R² = {r2_pop:.3f})")
    axes[1, 0].set_xlabel("Actual (log scale)")
    axes[1, 0].set_ylabel("Predicted (log scale)")
    axes[1, 0].legend(fontsize=9)

    decade_damage = (
        filtered.groupby(["Decade", "Disaster Type"])["Economic_Damage"]
        .mean().unstack(fill_value=0) / 1e9
    )
    plot_colors = [color_map[c] for c in decade_damage.columns if c in color_map]
    decade_damage.plot(kind="bar", ax=axes[1, 1],
                       color=plot_colors, edgecolor="white", rot=45)
    axes[1, 1].set_title("Avg. Economic Damage by Decade (Billion US$)")
    axes[1, 1].set_xlabel("Decade")
    axes[1, 1].set_ylabel("Avg. Damage (Billion US$)")
    axes[1, 1].legend(fontsize=8, title="Disaster Type")

    plt.tight_layout()
    plt.show()

min_year = int(impact_df["Start Year"].min())
max_year = int(impact_df["Start Year"].max())

year_range_slider = widgets.IntRangeSlider(
    value=[1970, 2021],
    min=min_year,
    max=max_year,
    step=1,
    description="Year Range:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

out = widgets.interactive_output(plot_impact_dashboard, {"year_range": year_range_slider})
display(year_range_slider, out)

NameError: name 'impact_df' is not defined